In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import difflib
from sklearn.metrics.pairwise import cosine_similarity



# Data Collection and Preprocessing

In [3]:
movies=pd.read_csv('movies.csv')

In [4]:
movies.head()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [5]:
movies.shape

(4803, 24)

In [6]:
movies.columns

Index(['index', 'budget', 'genres', 'homepage', 'id', 'keywords',
       'original_language', 'original_title', 'overview', 'popularity',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title',
       'vote_average', 'vote_count', 'cast', 'crew', 'director'],
      dtype='object')

In [7]:
#selecting important features for recommendation

imp_features=['genres','keywords','overview','tagline','cast','director']
print(imp_features)

['genres', 'keywords', 'overview', 'tagline', 'cast', 'director']


In [8]:
movies[imp_features].isnull().sum()

genres       28
keywords    412
overview      3
tagline     844
cast         43
director     30
dtype: int64

In [9]:
# Replacing the null values with null strings

movies[imp_features] = movies[imp_features].fillna('')

In [10]:
movies[imp_features].isnull().sum()

genres      0
keywords    0
overview    0
tagline     0
cast        0
director    0
dtype: int64

In [11]:
# combine all the imp_features

combined_features = (
    movies['genres'] + ' ' +
    movies['keywords'] + ' ' +
    movies['overview'] + ' ' +
    movies['cast'] + ' ' +
    movies['director'] + ' ' +
    movies['tagline']
)


In [12]:
combined_features

0       Action Adventure Fantasy Science Fiction cultu...
1       Adventure Fantasy Action ocean drug abuse exot...
2       Action Adventure Crime spy based on novel secr...
3       Action Crime Drama Thriller dc comics crime fi...
4       Action Adventure Science Fiction based on nove...
                              ...                        
4798    Action Crime Thriller united states\u2013mexic...
4799    Comedy Romance  A newlywed couple's honeymoon ...
4800    Comedy Drama Romance TV Movie date love at fir...
4801      When ambitious New York attorney Sam is sent...
4802    Documentary obsession camcorder crush dream gi...
Length: 4803, dtype: object

In [13]:
# converting the text data into feature vectors

vectorizer= TfidfVectorizer()

features_vector = vectorizer.fit_transform(combined_features)

In [14]:
print(features_vector)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 307355 stored elements and shape (4803, 30592)>
  Coords	Values
  (0, 561)	0.05971816344971169
  (0, 703)	0.06846420517510078
  (0, 9754)	0.08513696797398294
  (0, 23977)	0.07941905576010944
  (0, 10023)	0.07960231361105431
  (0, 6601)	0.1498786462809525
  (0, 5279)	0.1549075340655008
  (0, 10796)	0.11095111375730655
  (0, 25413)	0.24369151759694266
  (0, 29469)	0.08637114089261566
  (0, 5592)	0.17322386697661618
  (0, 25232)	0.1336739112380023
  (0, 13474)	0.03647840985795873
  (0, 27118)	0.08328687324810813
  (0, 239)	0.19716936546022962
  (0, 4768)	0.12501550204808315
  (0, 20104)	0.19260031442011954
  (0, 17021)	0.14324534514274753
  (0, 14023)	0.041700928584839175
  (0, 7827)	0.1797988202434295
  (0, 27405)	0.0310150192195571
  (0, 18249)	0.14769810641761003
  (0, 20039)	0.37747447361884223
  (0, 19541)	0.04908322232312576
  (0, 28597)	0.1549075340655008
  :	:
  (4802, 9588)	0.10415492266309316
  (4802, 21386)	0.0897551

# Cosine Similarity

In [15]:
# getting similarity score using Cosine_Similarity

similarity= cosine_similarity(features_vector)

In [16]:
print(similarity)

[[1.         0.05083168 0.0332947  ... 0.02749812 0.0304889  0.0072518 ]
 [0.05083168 1.         0.04356836 ... 0.05077045 0.03100979 0.01521198]
 [0.0332947  0.04356836 1.         ... 0.02646984 0.04751623 0.01372603]
 ...
 [0.02749812 0.05077045 0.02646984 ... 1.         0.03481447 0.03546821]
 [0.0304889  0.03100979 0.04751623 ... 0.03481447 1.         0.03098945]
 [0.0072518  0.01521198 0.01372603 ... 0.03546821 0.03098945 1.        ]]


In [17]:
similarity.shape

(4803, 4803)

In [18]:
# getting movie name by user

movie_name= input('Enter your favorites movie name :')

Enter your favorites movie name : iron man


In [19]:
# creating all list of movies name given in the dataset

list_of_all_titles = movies['title'].tolist()

In [20]:
# finding the close match of movie name given by user

find_close_match= difflib.get_close_matches(word=movie_name,possibilities=list_of_all_titles,n=3,cutoff=0.6)

In [21]:
find_close_match

['Iron Man', 'Iron Man 3', 'Iron Man 2']

In [22]:
close_match = find_close_match[0]

print(close_match)

Iron Man


In [23]:
# findung the index of movie using title

index_of_movie= movies[movies.title==close_match]['index'].values[0]

In [24]:
index_of_movie=int(index_of_movie)

In [25]:
index_of_movie

68

In [26]:
# Getting list of similar movie

similar_movie=list(enumerate(similarity[index_of_movie]))

In [27]:
len(similar_movie)

4803

In [28]:
# sorting of movies based on similarity score

sorted_similar_movie = sorted(similar_movie, key= lambda x:x[1],reverse=True)

In [29]:
# print name of simialar movie based on index

print('Movies suggested for you:/n')

i=1
for movie in sorted_similar_movie:
    index=movie[0]
    title_from_index = movies[movies['index'] == index]['title'].values[0]
    if(i<15):
        print(i,title_from_index)
        i+=1

Movies suggested for you:/n
1 Iron Man
2 Iron Man 2
3 Iron Man 3
4 Avengers: Age of Ultron
5 The Avengers
6 X-Men
7 The Helix... Loaded
8 Captain America: Civil War
9 X-Men: Apocalypse
10 Ant-Man
11 Made
12 Guardians of the Galaxy
13 X-Men: Days of Future Past
14 Super


# Movie Recommendation for the user input

In [30]:
movie_name= input('Enter your favorites movie name :')

list_of_all_titles = movies['title'].tolist()

find_close_match= difflib.get_close_matches(word=movie_name,possibilities=list_of_all_titles,n=3,cutoff=0.6)

close_match = find_close_match[0]

index_of_movie= movies[movies.title==close_match]['index'].values[0]

similar_movie=list(enumerate(similarity[index_of_movie]))

sorted_similar_movie = sorted(similar_movie, key= lambda x:x[1],reverse=True)

print('Top 10 Movies suggested for you:/n')

i=1
for movie in sorted_similar_movie:
    index=movie[0]
    title_from_index = movies[movies['index'] == index]['title'].values[0]
    if(i<11):
        print(i,title_from_index)
        i+=1

Enter your favorites movie name : iron man


Top 10 Movies suggested for you:/n
1 Iron Man
2 Iron Man 2
3 Iron Man 3
4 Avengers: Age of Ultron
5 The Avengers
6 X-Men
7 The Helix... Loaded
8 Captain America: Civil War
9 X-Men: Apocalypse
10 Ant-Man


In [33]:
import joblib

joblib.dump(similarity, 'similarity.pkl')

['similarity.pkl']

In [34]:
joblib.dump(movies, 'movies.pkl')

['movies.pkl']